# 台风中心追踪与移动速度诊断

本笔记本使用`局部最低压播种 + 迭代气压质心`算法，基于 `dataset/cm1out.nc` 逐时确定台风中心，
并绘制：
1. 台风中心轨迹图
2. 中心移动速度时间序列图

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from netCDF4 import Dataset

# ======================
# 用户参数
# ======================
nc_path = Path('dataset/cm1out.nc')

# 变量候选
ps_candidates = ['psfc', 'ps', 'prs', 'pres', 'slp', 'p_sfc']
x_dim = 'xh'
y_dim = 'yh'
time_dim = 'time'

# 追踪参数
search_radius = 80.0      # 局部最低压搜索半径
centroid_radius = 70.0    # 质心计算半径
max_iter = 8              # 每时次最大迭代次数
tol_grid = 0.01           # 收敛阈值（以网格间距为单位）

In [ ]:
def detect_ps_var(ds):
    for name in ps_candidates:
        if name in ds.variables:
            return name
    raise KeyError(f'未找到地面气压变量，候选={ps_candidates}')


def coord_as_2d(x1d, y1d):
    xx, yy = np.meshgrid(x1d.astype(float), y1d.astype(float))
    return xx, yy


def nearest_index_1d(arr, value):
    return int(np.argmin(np.abs(arr - value)))


def estimate_grid_spacing(x1d, y1d):
    dx = np.nanmedian(np.abs(np.diff(x1d.astype(float))))
    dy = np.nanmedian(np.abs(np.diff(y1d.astype(float))))
    vals = [v for v in [dx, dy] if np.isfinite(v) and v > 0]
    if not vals:
        return 1.0
    return float(np.mean(vals))


def local_minimum_seed(ps2d, xx, yy, center_x, center_y, radius):
    dist = np.hypot(xx - center_x, yy - center_y)
    mask = np.isfinite(ps2d) & (dist <= radius)
    if not np.any(mask):
        iy = nearest_index_1d(yy[:, 0], center_y)
        ix = nearest_index_1d(xx[0, :], center_x)
        return float(xx[iy, ix]), float(yy[iy, ix]), float(ps2d[iy, ix])

    idx_flat = int(np.argmin(np.where(mask, ps2d, np.inf)))
    iy, ix = np.unravel_index(idx_flat, ps2d.shape)
    return float(xx[iy, ix]), float(yy[iy, ix]), float(ps2d[iy, ix])


def pressure_centroid_iter(ps2d, xx, yy, seed_x, seed_y, radius, max_iter, tol_abs):
    curr_x, curr_y = float(seed_x), float(seed_y)
    for it in range(max_iter):
        dist = np.hypot(xx - curr_x, yy - curr_y)
        mask = np.isfinite(ps2d) & (dist <= radius)
        if not np.any(mask):
            return curr_x, curr_y, it + 1

        p_local = ps2d[mask]
        p_ref = float(np.nanmax(p_local))
        w = p_ref - p_local

        valid = np.isfinite(w) & (w > 0)
        if np.count_nonzero(valid) < 3:
            return curr_x, curr_y, it + 1

        x_local = xx[mask][valid]
        y_local = yy[mask][valid]
        w_local = w[valid]
        w_sum = float(np.sum(w_local))

        if not np.isfinite(w_sum) or w_sum <= 0:
            return curr_x, curr_y, it + 1

        new_x = float(np.sum(w_local * x_local) / w_sum)
        new_y = float(np.sum(w_local * y_local) / w_sum)

        shift = float(np.hypot(new_x - curr_x, new_y - curr_y))
        curr_x, curr_y = new_x, new_y
        if shift < tol_abs:
            return curr_x, curr_y, it + 1

    return curr_x, curr_y, max_iter


def normalize_time_to_hour(time_arr, time_units):
    t = np.asarray(time_arr)
    if np.issubdtype(t.dtype, np.datetime64):
        sec = (t - t[0]) / np.timedelta64(1, 's')
        return sec.astype(float) / 3600.0, 'datetime->hour'

    t = t.astype(float)
    if t.size < 2:
        return np.zeros_like(t), 'single-step'

    u = str(time_units or '').lower()
    if 'sec' in u:
        factor = 1.0 / 3600.0
        mode = f'units={time_units}'
    elif 'min' in u:
        factor = 1.0 / 60.0
        mode = f'units={time_units}'
    elif 'hour' in u or u.strip() in {'h', 'hr', 'hrs'}:
        factor = 1.0
        mode = f'units={time_units}'
    elif 'day' in u:
        factor = 24.0
        mode = f'units={time_units}'
    else:
        dt = np.median(np.diff(t))
        if dt > 1e11:
            factor, mode = 1.0 / 3.6e12, 'auto-ns'
        elif dt > 1e8:
            factor, mode = 1.0 / 3.6e9, 'auto-us'
        elif dt > 1e5:
            factor, mode = 1.0 / 3.6e6, 'auto-ms'
        elif dt > 1e2:
            factor, mode = 1.0 / 3600.0, 'auto-s'
        elif dt > 1:
            factor, mode = 1.0 / 60.0, 'auto-min'
        else:
            factor, mode = 1.0, 'auto-hour'

    return (t - t[0]) * factor, mode


def safe_gradient(values, t_hour):
    values = np.asarray(values, dtype=float)
    t_hour = np.asarray(t_hour, dtype=float)
    if len(values) < 2:
        return np.full_like(values, np.nan, dtype=float)
    if np.any(~np.isfinite(t_hour)) or np.any(np.diff(t_hour) <= 0):
        return np.full_like(values, np.nan, dtype=float)
    return np.gradient(values, t_hour)


def build_nc_candidates(primary_path: Path):
    candidates = [Path(primary_path)]
    # 中文路径失败时，尝试 ASCII 映射路径
    candidates.append(Path('C:/cm1_work/dataset/cm1out.nc'))
    unique = []
    seen = set()
    for p in candidates:
        k = str(p.resolve()) if p.exists() else str(p)
        if k not in seen:
            unique.append(p)
            seen.add(k)
    return unique


def load_arrays_with_fallback(primary_path: Path):
    last_err = None
    for p in build_nc_candidates(primary_path):
        if not p.exists():
            continue

        # 1) 优先 xarray 多引擎尝试
        for eng in (None, 'h5netcdf', 'scipy'):
            try:
                open_kwargs = {} if eng is None else {'engine': eng}
                with xr.open_dataset(p, **open_kwargs) as ds:
                    ps_name = detect_ps_var(ds)
                    xh = ds[x_dim].values.astype(float)
                    yh = ds[y_dim].values.astype(float)
                    ps_all = ds[ps_name].values.astype(float)
                    time_units = ds[time_dim].attrs.get('units', None) if time_dim in ds.variables else None
                    time_raw = ds[time_dim].values if time_dim in ds.variables else np.arange(ps_all.shape[0])
                used_engine = 'default(netcdf4)' if eng is None else eng
                return xh, yh, ps_all, time_raw, time_units, ps_name, str(p), used_engine
            except Exception as e:
                last_err = e

        # 2) xarray 全失败后，netCDF4 兜底读取
        try:
            with Dataset(p, 'r') as nc:
                var_names = set(nc.variables.keys())
                ps_name = next((v for v in ps_candidates if v in var_names), None)
                if ps_name is None:
                    raise KeyError(f'未找到地面气压变量，候选={ps_candidates}，实际={list(var_names)}')

                xh = np.asarray(nc.variables[x_dim][:], dtype=float)
                yh = np.asarray(nc.variables[y_dim][:], dtype=float)
                ps_all = np.asarray(nc.variables[ps_name][:], dtype=float)
                if time_dim in nc.variables:
                    time_var = nc.variables[time_dim]
                    time_raw = np.asarray(time_var[:])
                    time_units = getattr(time_var, 'units', None)
                else:
                    time_raw = np.arange(ps_all.shape[0])
                    time_units = None
            return xh, yh, ps_all, time_raw, time_units, ps_name, str(p), 'netCDF4-fallback'
        except Exception as e:
            last_err = e

    raise FileNotFoundError(f'无法读取数据文件。主路径={primary_path}，最后错误={last_err}')


xh, yh, ps_all, time_raw, time_units, ps_name, nc_used, read_mode = load_arrays_with_fallback(nc_path)

if ps_all.ndim < 3:
    raise ValueError(f'{ps_name} 维度不是(time,y,x)或可兼容形式，当前shape={ps_all.shape}')

# 默认取 time 后的最后两维为 (y, x)
nt = ps_all.shape[0]
ps_all = ps_all.reshape(nt, ps_all.shape[-2], ps_all.shape[-1])

if ps_all.shape[1:] == (len(xh), len(yh)):
    ps_all = np.transpose(ps_all, (0, 2, 1))
elif ps_all.shape[1:] != (len(yh), len(xh)):
    raise ValueError(f'{ps_name} 最后两维与 xh/yh 不匹配: {ps_all.shape[1:]} vs ({len(yh)}, {len(xh)})')

xx, yy = coord_as_2d(xh, yh)
dxy = estimate_grid_spacing(xh, yh)
tol_abs = tol_grid * dxy

records = []
center_prev = None
domain_radius = float(max(np.ptp(xh), np.ptp(yh))) * 2.0
if not np.isfinite(domain_radius) or domain_radius <= 0:
    domain_radius = max(search_radius, centroid_radius)

for t_idx in range(nt):
    ps2d = ps_all[t_idx]

    if center_prev is None:
        seed_center_x = float(np.nanmean(xh))
        seed_center_y = float(np.nanmean(yh))
        seed_search_radius = domain_radius
    else:
        seed_center_x, seed_center_y = center_prev
        seed_search_radius = search_radius

    seed_x, seed_y, seed_p = local_minimum_seed(ps2d, xx, yy, seed_center_x, seed_center_y, seed_search_radius)
    c_x, c_y, n_iter = pressure_centroid_iter(
        ps2d, xx, yy, seed_x, seed_y, centroid_radius, max_iter, tol_abs
    )

    center_prev = (c_x, c_y)
    records.append({
        'time_index': t_idx,
        'time_raw': time_raw[t_idx],
        'center_x': c_x,
        'center_y': c_y,
        'seed_x': seed_x,
        'seed_y': seed_y,
        'seed_p': seed_p,
        'n_iter': n_iter,
    })

df = pd.DataFrame(records)
df['time_hour'], time_mode = normalize_time_to_hour(df['time_raw'].to_numpy(), time_units)

u = safe_gradient(df['center_x'].to_numpy(), df['time_hour'].to_numpy())
v = safe_gradient(df['center_y'].to_numpy(), df['time_hour'].to_numpy())
df['u_trans'] = u
df['v_trans'] = v
df['V_trans'] = np.hypot(u, v)

print(f'读取路径: {nc_used}')
print(f'读取模式: {read_mode}')
print(f'识别变量: {ps_name}')
print(f'时间换算模式: {time_mode}')
print(df.head())
df

FileNotFoundError: [Errno 2] No such file or directory: 'e:\\科研\\数值模式cm1\\dataset\\cm1out.nc'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=130)

ax0 = axes[0]
ax0.plot(df['center_x'], df['center_y'], '-o', ms=2.8, lw=1.4, label='TC center')
ax0.scatter(df['center_x'].iloc[0], df['center_y'].iloc[0], marker='s', s=45, label='start')
ax0.scatter(df['center_x'].iloc[-1], df['center_y'].iloc[-1], marker='^', s=55, label='end')
ax0.set_xlabel('x')
ax0.set_ylabel('y')
ax0.set_title('Typhoon Center Track')
ax0.axis('equal')
ax0.grid(True, ls='--', alpha=0.35)
ax0.legend(fontsize=8)

ax1 = axes[1]
ax1.plot(df['time_hour'], df['u_trans'], lw=1.2, label='u_trans (dx/dt)')
ax1.plot(df['time_hour'], df['v_trans'], lw=1.2, label='v_trans (dy/dt)')
ax1.plot(df['time_hour'], df['V_trans'], lw=1.8, label='speed')
ax1.set_xlabel('time (hour)')
ax1.set_ylabel('translation speed (coord unit / hour)')
ax1.set_title('Center Translation Speed')
ax1.grid(True, ls='--', alpha=0.35)
ax1.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
out_csv = Path('output_center_track_from_cm1out.csv')
df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'已保存中心与速度结果: {out_csv.resolve()}')